In [7]:
import numpy as np
from src.analysis.processing import lime_ranking, shap_ranking, shapiq_ranking, meg_ranking, mmace_ranking, \
    meg_cf_percent, mmace_cf_percent
from src.analysis.xai_eval import pgi, pgu
import pickle
import os
import joblib

max_k = 6
dataset_name = 'qm9_nonlinear_6'
results_dir = f'../results/synthetic_data/{dataset_name}/explanations'
model_dir = f'../results/synthetic_data/{dataset_name}/'

target = 'target'

results_dict = {
    'lime': ('lime_results.pickle', lime_ranking),
    'shap': ('shap_results.pickle', shap_ranking),
    'shapiq1': ('shapiq1_results.pickle', shapiq_ranking),
    'shapiq2': ('shapiq2_results.pickle', shapiq_ranking),
    'meg': ('meg_results.pickle', meg_ranking, meg_cf_percent),
    'mmace': ('mmace_results.pickle', mmace_ranking, mmace_cf_percent),
}

ranking_dict = {}
ranking_per_fold_dict = {}
cf_similarity_dict = {}
cf_validity_dict = {}
metrics_dict = {}
metrics_top10_dict = {}

for key in results_dict.keys():
    print(key)
    file_name, ranking_func = results_dict[key][:2]

    if len(results_dict[key]) > 2:
        cf_func = results_dict[key][2]
    else:
        cf_func = None
    with open(os.path.join(results_dir, file_name), 'rb') as f:
        results = pickle.load(f)

    ranking, rankings_per_fold = ranking_func(results, target)
    if cf_func is not None:
        cf_percent = cf_func(results, target=target)
    else:
        cf_percent = None

    if cf_percent is not None:
        print(f"{key} counterfactual percent: {cf_percent}")

    pgis, pgus = [], []
    pgis_10, pgus_10 = [], []
    pgis_org, pgus_org = [], []
    pgis_org_10, pgus_org_10 = [], []

    ranking_dict[key] = ranking
    ranking_per_fold_dict[key] = rankings_per_fold
    if cf_percent is not None:
        cf_validity_dict[key] = cf_percent[0]
        cf_similarity_dict[key] = cf_percent[1]

    for i in range(len(rankings_per_fold)):
        model = os.path.join(model_dir, f'model_{i}.joblib')
        model = joblib.load(model)
        test_examples = results['test_data'][i].drop(columns=[target])
        train_examples = results['training_data'][i].drop(columns=[target])

        ranking_current = list(rankings_per_fold[i]['features'])

        pgi_one, pgi_org = pgi(test_examples, ranking_current, model, train_examples)
        pgu_one, pgu_org = pgu(test_examples, ranking_current, model, train_examples)
        pgi_ten, pgi_ten_org = pgi(test_examples, ranking_current, model, train_examples, len_max=max_k)
        pgu_ten, pgu_ten_org = pgu(test_examples, ranking_current, model, train_examples, len_max=max_k)
        pgis.append(pgi_one)
        pgus.append(pgu_one)
        pgis_10.append(pgi_ten)
        pgus_10.append(pgu_ten)
        pgis_org.append(pgi_org)
        pgus_org.append(pgu_org)
        pgis_org_10.append(pgi_ten_org)
        pgus_org_10.append(pgu_ten_org)

    pgi_mean = np.mean(pgis)
    pgu_mean = np.mean(pgus)
    pgi_std = np.std(pgis)
    pgu_std = np.std(pgus)
    pgi_org_mean = np.mean(pgis_org)
    pgu_org_mean = np.mean(pgus_org)
    pgi_org_std = np.std(pgis_org)
    pgu_org_std = np.std(pgus_org)
    pgi_mean_10 = np.mean(pgis_10)
    pgu_mean_10 = np.mean(pgus_10)
    pgi_std_10 = np.std(pgis_10)
    pgu_std_10 = np.std(pgus_10)
    pgi_org_mean_10 = np.mean(pgis_org_10)
    pgu_org_mean_10 = np.mean(pgus_org_10)
    pgi_org_std_10 = np.std(pgis_org_10)
    pgu_org_std_10 = np.std(pgus_org_10)
    metrics_dict[key] = {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        f'pgi_mean_{max_k}': pgi_mean_10,
        f'pgu_mean_{max_k}': pgu_mean_10,
        f'pgi_std_{max_k}': pgi_std_10,
        f'pgu_std_{max_k}': pgu_std_10,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
        f'pgi_org_mean_{max_k}': pgi_org_mean_10,
        f'pgu_org_mean_{max_k}': pgu_org_mean_10,
        f'pgi_org_std_{max_k}': pgi_org_std_10,
        f'pgu_org_std_{max_k}': pgu_org_std_10,
    }

    print(f"{key} PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
    print(f"PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")
    print(f"PGI 10: {pgi_mean_10} ({pgi_std_10}), PGU 10: {pgu_mean_10} ({pgu_std_10})")
    print(f"PGI Org 10: {pgi_org_mean_10} ({pgi_org_std_10}), PGU Org 10: {pgu_org_mean_10} ({pgu_org_std_10})")

    print('--' * 20)

lime
lime PGI: 0.5212240446954617 (0.08839274248527071), PGU: 0.06028303271637921 (0.01456982206258415)
PGI Org: 13.328204621748915 (0.9139782609579599), PGU Org: 1.543752502109942 (0.33417156128306547)
PGI 10: 0.5281979473391232 (0.10969322652505073), PGU 10: 0.0443434694680569 (0.013025029073265436)
PGI Org 10: 13.416594970152289 (0.9131492722956648), PGU Org 10: 1.1337100449147974 (0.31137297891679755)
----------------------------------------
shap
shap PGI: 0.5152071000514752 (0.08674797645076363), PGU: 0.05464914478909899 (0.012180228201816778)
PGI Org: 13.18153405214944 (0.9396099668254702), PGU Org: 1.3991006904575327 (0.2573642593875208)
PGI 10: 0.5231605120512894 (0.10678843711626604), PGU 10: 0.03846556504120325 (0.009842674707305639)
PGI Org 10: 13.301753827358937 (0.9471534569193302), PGU Org 10: 0.9833642827039186 (0.2144853695466829)
----------------------------------------
shapiq1
shapiq1 PGI: 0.513497082689889 (0.09053948676015797), PGU: 0.061596468960482806 (0.011895415

In [8]:
import pandas as pd
def convert_term_ranking_to_feature_ranking(ranking_with_interactions: list) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    feature_to_best_rank = {}
    for i, term in enumerate(ranking_with_interactions):
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = i

    # Sort the features by their best rank
    sorted_features = sorted(feature_to_best_rank.items(), key=lambda item: item[1])

    return [feature for feature, rank in sorted_features]

def aggregate_rankings_by_mean_position(list_of_rankings: list) -> list:
    if not list_of_rankings:
        return []
    all_items = set()
    for ranking in list_of_rankings:
        all_items.update(ranking)
    item_scores = {}
    for item in all_items:
        positions = []
        for ranking in list_of_rankings:
            try:
                position = ranking.index(item)
            except ValueError:
                position = len(ranking)
            positions.append(position)
        item_scores[item] = np.mean(positions)
    sorted_items = sorted(item_scores.keys(), key=lambda item: item_scores[item])
    return sorted_items

In [9]:
pgis, pgus = [], []
pgis_org, pgus_org = [], []
pgis_org_10, pgus_org_10 = [], []
pgis_10, pgus_10 = [], []
for i in range(len(ranking_per_fold_dict['lime'])):

    model = os.path.join(model_dir, f'model_{i}.joblib')
    model = joblib.load(model)
    test_examples = results['test_data'][i].drop(columns=[target])
    train_examples = results['training_data'][i].drop(columns=[target])

    rankings = []
    for key in ranking_per_fold_dict.keys():
        ranking_current = list(ranking_per_fold_dict[key][i]['features'])
        if key == 'shapiq2':
            ranking_current = convert_term_ranking_to_feature_ranking(ranking_current)
        rankings.append(ranking_current)

    aggregated_ranking = aggregate_rankings_by_mean_position(rankings)
    pgi_one, pgi_org = pgi(test_examples, aggregated_ranking, model, train_examples)
    pgu_one, pgu_org = pgu(test_examples, aggregated_ranking, model, train_examples)
    pgi_ten, pgi_ten_org = pgi(test_examples, aggregated_ranking, model, train_examples, len_max=max_k)
    pgu_ten, pgu_ten_org = pgu(test_examples, aggregated_ranking, model, train_examples, len_max=max_k)
    pgis.append(pgi_one)
    pgus.append(pgu_one)
    pgis_10.append(pgi_ten)
    pgus_10.append(pgu_ten)
    pgis_org.append(pgi_org)
    pgus_org.append(pgu_org)
    pgis_org_10.append(pgi_ten_org)
    pgus_org_10.append(pgu_ten_org)
pgi_mean = np.mean(pgis)
pgu_mean = np.mean(pgus)
pgi_std = np.std(pgis)
pgu_std = np.std(pgus)
pgi_org_mean = np.mean(pgis_org)
pgu_org_mean = np.mean(pgus_org)
pgi_org_std = np.std(pgis_org)
pgu_org_std = np.std(pgus_org)
pgi_mean_10 = np.mean(pgis_10)
pgu_mean_10 = np.mean(pgus_10)
pgi_std_10 = np.std(pgis_10)
pgu_std_10 = np.std(pgus_10)
pgi_org_mean_10 = np.mean(pgis_org_10)
pgu_org_mean_10 = np.mean(pgus_org_10)
pgi_org_std_10 = np.std(pgis_org_10)
pgu_org_std_10 = np.std(pgus_org_10)

metrics_dict['aggregated'] = {
        'pgi_mean': pgi_mean,
        'pgu_mean': pgu_mean,
        'pgi_std': pgi_std,
        'pgu_std': pgu_std,
        f'pgi_mean_{max_k}': pgi_mean_10,
        f'pgu_mean_{max_k}': pgu_mean_10,
        f'pgi_std_{max_k}': pgi_std_10,
        f'pgu_std_{max_k}': pgu_std_10,
        'pgi_org_mean': pgi_org_mean,
        'pgu_org_mean': pgu_org_mean,
        'pgi_org_std': pgi_org_std,
        'pgu_org_std': pgu_org_std,
        f'pgi_org_mean_{max_k}': pgi_org_mean_10,
        f'pgu_org_mean_{max_k}': pgu_org_mean_10,
        f'pgi_org_std_{max_k}': pgi_org_std_10,
        f'pgu_org_std_{max_k}': pgu_org_std_10,
    }

print(f"Aggregated PGI: {pgi_mean} ({pgi_std}), PGU: {pgu_mean} ({pgu_std})")
print(f"Aggregated PGI Org: {pgi_org_mean} ({pgi_org_std}), PGU Org: {pgu_org_mean} ({pgu_org_std})")
print(f"Aggregated PGI 10: {pgi_mean_10} ({pgi_std_10}), PGU 10: {pgu_mean_10} ({pgu_std_10})")
print(f"Aggregated PGI Org 10: {pgi_org_mean_10} ({pgi_org_std_10}), PGU Org 10: {pgu_org_mean_10} ({pgu_org_std_10})")

Aggregated PGI: 0.517119711807953 (0.08889054757494588), PGU: 0.061903698142985894 (0.013862959242836993)
Aggregated PGI Org: 13.221888814631566 (0.9413014834417209), PGU Org: 1.5777085529002428 (0.25649393311477703)
Aggregated PGI 10: 0.5291890357738442 (0.114587760712076), PGU 10: 0.04315744505876437 (0.011221208698040434)
Aggregated PGI Org 10: 13.423009216839919 (1.003177237951579), PGU Org 10: 1.0941251466307766 (0.2055294029532347)


In [10]:
# Save the results
os.makedirs(os.path.join(results_dir, 'analysis'), exist_ok=True)
with open(os.path.join(results_dir, 'analysis', 'metrics_results.pickle'), 'wb') as f:
    pickle.dump(metrics_dict, f)
with open(os.path.join(results_dir, 'analysis', 'metrics_top10_results.pickle'), 'wb') as f:
    pickle.dump(metrics_top10_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_results.pickle'), 'wb') as f:
    pickle.dump(ranking_dict, f)
with open(os.path.join(results_dir, 'analysis', 'ranking_per_fold_results.pickle'), 'wb') as f:
    pickle.dump(ranking_per_fold_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_validity_results.pickle'), 'wb') as f:
    pickle.dump(cf_validity_dict, f)
with open(os.path.join(results_dir, 'analysis', 'cf_similarity_results.pickle'), 'wb') as f:
    pickle.dump(cf_similarity_dict, f)

In [11]:
from src.analysis.xai_eval import rank_correlation

def convert_term_ranking_to_feature_ranking(ranking_with_interactions: list) -> list:
    """
    Converts a ranking of terms (features + interactions) into a ranking of
    only features based on their earliest appearance.
    """
    feature_to_best_rank = {}
    for i, term in enumerate(ranking_with_interactions):
        constituent_features = term.split(' x ')
        for feature in constituent_features:
            if feature not in feature_to_best_rank:
                feature_to_best_rank[feature] = i

    # Sort the features by their best rank
    sorted_features = sorted(feature_to_best_rank.items(), key=lambda item: item[1])

    return [feature for feature, rank in sorted_features]

#calculate rankings correlations
pairs_of_ranks = [(key1, key2) for key1 in ranking_per_fold_dict.keys() for key2 in ranking_per_fold_dict.keys()]
correlations = {}
for key1, key2 in pairs_of_ranks:
    corrs = []
    for i in range(len(ranking_per_fold_dict[key1])):
        rank1 = ranking_per_fold_dict[key1][i]['features']
        rank2 = ranking_per_fold_dict[key2][i]['features']
        if key1 == 'shapiq2':
            rank1 = convert_term_ranking_to_feature_ranking(rank1)
        if key2 == 'shapiq2':
            rank2 = convert_term_ranking_to_feature_ranking(rank2)
        c = rank_correlation(rank1, rank2).statistic
        corrs.append(c)
    correlation = np.mean(corrs)
    correlations[(key1, key2)] = correlation
    print(f"{key1} vs {key2}: {correlation:.4f}")
    print('--' * 20)

with open(os.path.join(results_dir, 'analysis', 'correlations_results.pickle'), 'wb') as f:
    pickle.dump(correlations, f)

lime vs lime: 1.0000
----------------------------------------
lime vs shap: 0.7764
----------------------------------------
lime vs shapiq1: 0.6666
----------------------------------------
lime vs shapiq2: 0.6629
----------------------------------------
lime vs meg: 0.2296
----------------------------------------
lime vs mmace: 0.3226
----------------------------------------
shap vs lime: 0.7764
----------------------------------------
shap vs shap: 1.0000
----------------------------------------
shap vs shapiq1: 0.8304
----------------------------------------
shap vs shapiq2: 0.7921
----------------------------------------
shap vs meg: 0.4030
----------------------------------------
shap vs mmace: 0.4973
----------------------------------------
shapiq1 vs lime: 0.6666
----------------------------------------
shapiq1 vs shap: 0.8304
----------------------------------------
shapiq1 vs shapiq1: 1.0000
----------------------------------------
shapiq1 vs shapiq2: 0.9385
-------------------

In [12]:
#calculate rankings correlations
pairs_of_ranks = [(key1, key2) for key1 in ranking_per_fold_dict.keys() for key2 in ranking_per_fold_dict.keys()]
correlations_top10 = {}
for key1, key2 in pairs_of_ranks:
    corrs = []
    for i in range(len(ranking_per_fold_dict[key1])):
        rank1 = ranking_per_fold_dict[key1][i]['features']
        rank2 = ranking_per_fold_dict[key2][i]['features']
        if key1 == 'shapiq2':
            rank1 = convert_term_ranking_to_feature_ranking(rank1)
        if key2 == 'shapiq2':
            rank2 = convert_term_ranking_to_feature_ranking(rank2)
        c = rank_correlation(rank1, rank2, k=max_k).statistic
        corrs.append(c)
    correlation = np.mean(corrs)
    correlations_top10[(key1, key2)] = correlation
    print(f"{key1} vs {key2}: {correlation:.4f}")
    print('--' * 20)

with open(os.path.join(results_dir, 'analysis', f'correlations_top{max_k}_results.pickle'), 'wb') as f:
    pickle.dump(correlations_top10, f)

lime vs lime: 1.0000
----------------------------------------
lime vs shap: 0.8414
----------------------------------------
lime vs shapiq1: 0.5780
----------------------------------------
lime vs shapiq2: 0.5337
----------------------------------------
lime vs meg: 0.2139
----------------------------------------
lime vs mmace: 0.3142
----------------------------------------
shap vs lime: 0.8414
----------------------------------------
shap vs shap: 1.0000
----------------------------------------
shap vs shapiq1: 0.6880
----------------------------------------
shap vs shapiq2: 0.6319
----------------------------------------
shap vs meg: 0.0915
----------------------------------------
shap vs mmace: 0.1896
----------------------------------------
shapiq1 vs lime: 0.5780
----------------------------------------
shapiq1 vs shap: 0.6880
----------------------------------------
shapiq1 vs shapiq1: 1.0000
----------------------------------------
shapiq1 vs shapiq2: 0.9386
-------------------